# 🌐 Aula 20 — Gêmeos Digitais, IIoT e IA Generativa

**Disciplina:** IA Aplicada à Engenharia Química  
**Dataset:** trocador_e101_24h.csv — 24 h do trocador com incrustação

---

## Conceito

Um gêmeo digital (digital twin) = modelo + dados em tempo real + loop de atualização + capacidade de simulação. "O espelho digital da planta."

- Sensor 4-20 mA -> IIoT (OPC-UA/MQTT) -> edge -> cloud -> modelo -> ação


## Validação: Detecção de Drift

O twin assume U_limpo=800 fixo. Com incrustação, U cai para ~400 e o twin começa a errar T_saida. Drift = erro persistente:

```
MAE_janela(t) > θ  por  Δt > τ  ->  alerta de re-treino
```
(θ ~ 5% da medição, τ ~ 1 h)


## 3.1 — Exercício Guiado: Detectar Drift do Twin


### Passo 1: carregar dados do trocador


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula20/trocador_e101_24h.csv"
df = pd.read_csv(URL, parse_dates=['timestamp'])
df.head()


### Passo 2: U real (drift de incrustação)


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(df['U_real_W_m2K'], color='steelblue')
plt.xlabel('minuto'); plt.ylabel('U (W/m2K)')
plt.title('Incrustacao: U cai de ~800 para ~400')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


### Passo 3: T_saida real vs twin (drift visível)


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(df['T_saida_real_C'], label='T_saida real')
plt.plot(df['T_saida_twin_C'], '--', label='T_saida twin (U fixo)')
plt.xlabel('minuto'); plt.ylabel('C')
plt.legend(); plt.grid(alpha=0.3); plt.title('Drift: twin vs planta')
plt.tight_layout(); plt.show()


### Passo 4: calcular MAE em janelas de 1h


In [ ]:
N = 60  # janela 1h
mae = df['T_saida_real_C'].rolling(N).apply(
      lambda w: np.mean(np.abs(w - df['T_saida_twin_C'].loc[w.index])), raw=False)
df['MAE_1h'] = mae

theta = 0.05*df['T_saida_real_C'].mean()   # 5% da medicao
print(f"Limiar theta = {theta:.2f} C (5%)")

plt.figure(figsize=(10,4))
plt.plot(df['MAE_1h'], color='orange')
plt.axhline(theta, color='red', ls='--', label=f'Limiar {theta:.2f}')
plt.xlabel('minuto'); plt.ylabel('MAE twin (C)')
plt.legend(); plt.grid(alpha=0.3); plt.title('MAE - quando o drift cruza o limiar')
plt.tight_layout(); plt.show()


### Passo 5: regra de alerta persistente (theta, tau=1h)


In [ ]:
# Regra: alerta se MAE > theta por periodo sustentado (tau ~ 1h)
df['alerta'] = (df['MAE_1h'] > theta).astype(int)
cruzamento = df.index[df['alerta'].diff().fillna(0) > 0]
print("Primeiro cruzamento do limiar (min):", cruzamento.min() if len(cruzamento) else 'nenhum')
print(f"Minutos em alerta: {df['alerta'].sum()} de {len(df)}")
print("Conclusao: drift de incrustacao detectado => retreinar o U do twin")


## 3.2 — Exercício em Grupo: Projetar DT + LLM

1. Cada grupo projeta o DT de um equipamento (reator, coluna, bomba, compressor)
2. Preencha: sensores | protocolo | modelo | frequência | drift esperado
3. **LLM:** gere um parágrafo de diagnóstico dos dados e avalie criticamente
   (alucinou? correto? útil?) — sempre com **revisão humana** (anti-alucinação)


## Checklist projeto de DT

- [ ] Sensores listados (T, P, vazão, dP)
- [ ] Protocolos IIoT (OPC-UA / MQTT)
- [ ] Modelo escolhido (híbrido ML+balanço)
- [ ] Frequência de atualização/re-treino
- [ ] Validação por drift (theta / tau)
- [ ] Dashboard (real vs twin, U estimado)
- [ ] Revisão humana do LLM
